# X-Guard v1: YOLOv8n foreign-object training on Colab

Trains YOLOv8n to find foreign objects in 850 nm near-infrared photos taken through the product (ELP OV9281, 1280x800 grayscale).
The labels are `stone`, `metal` and `plastic`, but by default (`merge='foreign'`) they are merged into a single `foreign` class. Photos with no boxes are normal products.
Data conversion, augmentation, training, the live mAP plot, history plots and saving are all functions in this notebook,
so all you need on Colab is this file and the data zips on Drive.

How to use it
1. Make a folder on Google Drive and put the data zips in it (e.g. `My Drive/xguard/`):
   - `Xguard_defect.zip`: CVAT "YOLO 1.1" labels (`obj.names`, `obj_train_data/*.txt`)
   - `defect.zip`: images (`defect/*.png`)
   - Unzipped folders work too.
2. In Colab, Runtime > Change runtime type > T4 GPU.
3. Set `ROOT` in the settings cell below to your folder, then Runtime > Run all.
4. When it's done, `ROOT/model/` has `detect.pt`, `classes.txt` and `model_info.json`.
   Copy them into `models/` in the project and the conveyor AI (`ai/app.py`) will use the new model.

| Step | Function |
|---|---|
| Unzip the data | `unzip_all(root, work)` |
| Build the YOLO folders (7:1.5:1.5 split per class) | `prepare_custom(src, out)` |
| Dataset stats plot | `plot_dataset(data_dir, names)` |
| Train, with live loss / P / R / mAP50 / mAP50-95 per epoch | `train(cfg)` |
| Test evaluation | `evaluate(model, cfg, data_yaml, name)` |
| Save for the conveyor | `save_for_conveyor(...)` |
| All of the above | `run(CFG)` |
| History plots | `plot_history` `plot_best` `plot_per_class` `show_val_curves` `show_predictions` |

## 0. Environment

In [ ]:
%pip install -q "ultralytics>=8.3" "albumentations>=2.0"

In [ ]:
# Check for a GPU. If False: Runtime > Change runtime type > T4 GPU, save, and run from the top again.
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU 없음 (CPU 로 학습, 아주 느림)')

In [ ]:
import os
import csv
import json
import glob
import random
import shutil
import zipfile
import hashlib
from datetime import datetime

import yaml
import matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, display

IN_COLAB = os.path.exists('/content')
if IN_COLAB and not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

## 1. Settings (the only cell you need to edit)

`ROOT` is the Drive folder with the data zips. Results are saved to `ROOT/model/`.

In [ ]:
ROOT = '/content/drive/MyDrive/xguard'   # <- your Drive folder

CFG = dict(
    root=ROOT,
    work_dir='/content/xguard_work',      # unzip and convert here (Colab's local disk is much faster than Drive)
    out_dir=os.path.join(ROOT, 'model'),  # final model goes here (Drive)

    # model
    model='yolov8n',                      # yolov8n (default) / yolov8s / yolo11n ...
    imgsz=640,                            # ai/app.py picks this up from model_info.json
    # train
    epochs=100,
    batch=16,                             # 8 if the GPU runs out of memory
    optimizer='auto',
    lr=0.01,                              # ignored when optimizer is auto
    patience=20,                          # early stop
    device='auto',                        # auto = GPU 0 if there is one, else cpu (with a warning). Or set 0 / 'cpu'
    workers=2,                            # Colab has 2 CPUs
    seed=0,
    # aug
    aug='nir',                            # default / none / xray / nir / strong
    mosaic=1.0,                           # 0 = off
    # data split (train / valid / test)
    split=(0.7, 0.15, 0.15),
    split_seed=42,
    # any other ultralytics train argument, e.g. dict(cos_lr=True, dropout=0.1, fliplr=0.5)
    opt=dict(),
    name=None,                            # default: {model}_{aug}_{time}
    model_ver='yolov8n-xguard-1.0',       # modelVer shown on the dashboard
    live=True,                            # redraw the plot every epoch
    # class: 'foreign' merges stone/metal/plastic into one class, None keeps all three
    merge='foreign',
)

## 2. Data functions (our data -> ultralytics layout)
Labels and images can be in different folders, as in the CVAT export; they are paired by file name.
The dataset is small, so the split is done per class to make sure valid and test get every class.

In [ ]:
# unzip any zips into work_dir; with no zips, use the root folder as the dataset
def unzip_all(root, work_dir):
    zips = sorted(glob.glob(os.path.join(root, '*.zip')))
    if not zips:
        print('zip 없음 -> 폴더를 그대로 사용:', root)
        return root
    src = os.path.join(work_dir, 'raw')
    shutil.rmtree(src, ignore_errors=True)
    os.makedirs(src)
    for z in zips:
        zipfile.ZipFile(z).extractall(src)
        print('unzip:', os.path.basename(z))
    return src


# our own dataset (YOLO txt labels) -> ultralytics folder
# (only standard library + yaml -> can be checked before torch / ultralytics are installed)
#
# accepted layouts (dataset root):
#   A) images/ + labels/                                  -> split automatically (SPLIT)
#      (labels can also sit next to the images: a.jpg + a.txt)
#   B) train/images, train/labels, valid(or val)/..., test/...   (Roboflow 'YOLOv8' export)
#   C) img/{train,valid,test} + labels/{train,valid,test}         (this repo's dataset/ folder)
#   D) CVAT 'YOLO 1.1' export: obj.names, obj.data, train.txt, obj_train_data/*.txt
#      + images anywhere (e.g. defect/*.png) -> matched by file name -> split like A
#   no valid -> 15% of train, no test -> test = valid
#
# class names: classes.txt / obj.names (one per line, line 0 = class 0) or data.yaml 'names'
# image without label file (or empty file) = normal product (background) -> fine, keep them
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp')
SPLIT = (0.7, 0.15, 0.15)   # train / valid / test (layout A)
SEED = 42
SPLITS = ('train', 'valid', 'test')
SPLIT_ALIAS = {'train': 'train', 'valid': 'valid', 'val': 'valid', 'test': 'test'}
NAME_FILES = ('classes.txt', 'obj.names')                      # class names, one per line
NOT_LABELS = NAME_FILES + ('train.txt', 'valid.txt', 'val.txt', 'test.txt')   # CVAT image lists


def is_image(path):
    return path.lower().endswith(IMG_EXTS)


# every image under folder (sub folders too), sorted
def list_images(folder):
    out = []
    for root, _, files in os.walk(folder):
        out += [os.path.join(root, f) for f in files if is_image(f)]
    return sorted(out)


# number of images in the dataset folder (0 = nothing to train)
def count_images(src):
    if not os.path.isdir(src):
        return 0
    return len(list_images(src))


# every label txt under folder: file name (no ext) -> path
# (CVAT keeps labels in obj_train_data/ and images somewhere else)
def label_index(folder):
    out = {}
    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith('.txt') and f not in NOT_LABELS:
                out.setdefault(os.path.splitext(f)[0], os.path.join(root, f))
    return out


# image path -> label path or None
#   .../images/x.jpg -> .../labels/x.txt  (also img/ -> labels/)
#   .../x.jpg        -> .../x.txt         (labelImg default)
#   index[x]                              (same file name anywhere, CVAT)
def find_label(img, index=None):
    stem = os.path.splitext(img)[0]
    parts = stem.replace('\\', '/').split('/')
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] in ('images', 'img'):
            p = '/'.join(parts[:i] + ['labels'] + parts[i + 1:]) + '.txt'
            if os.path.exists(p):
                return p
    if os.path.exists(stem + '.txt') and os.path.basename(stem) + '.txt' not in NOT_LABELS:
        return stem + '.txt'
    return (index or {}).get(os.path.basename(stem))


# classes.txt / obj.names or data.yaml -> ['stone', 'metal', ...]
def read_class_names(src):
    for root, _, files in os.walk(src):
        for name in NAME_FILES:
            if name in files:
                with open(os.path.join(root, name), encoding='utf-8') as f:
                    return [l.strip() for l in f if l.strip()]
    for root, _, files in os.walk(src):
        for name in ('data.yaml', 'dataset.yaml'):
            if name in files:
                with open(os.path.join(root, name), encoding='utf-8') as f:
                    names = (yaml.safe_load(f) or {}).get('names')
                if isinstance(names, dict):
                    return [names[k] for k in sorted(names)]
                if names:
                    return list(names)
    return []


# dataset root -> {'train': [...], 'valid': [...], 'test': [...]} or {None: [...]} (layout A, not split yet)
def collect(src):
    split = {}
    for d in os.listdir(src):                       # B) train/ valid/ test/
        if d.lower() in SPLIT_ALIAS and os.path.isdir(os.path.join(src, d)):
            split.setdefault(SPLIT_ALIAS[d.lower()], []).extend(list_images(os.path.join(src, d)))
    for top in ('img', 'images'):                   # C) img/train ...
        folder = os.path.join(src, top)
        if not os.path.isdir(folder):
            continue
        for d in os.listdir(folder):
            if d.lower() in SPLIT_ALIAS and os.path.isdir(os.path.join(folder, d)):
                split.setdefault(SPLIT_ALIAS[d.lower()], []).extend(list_images(os.path.join(folder, d)))
    if split.get('train'):
        return split
    return {None: list_images(src)}                 # A) everything


# list -> train / valid / test by ratio (at least 1 valid / 1 test when there are >= 3 images)
def split_list(items, split=SPLIT, seed=SEED):
    items = sorted(items)
    random.Random(seed).shuffle(items)
    n = len(items)
    n_val = max(1, round(n * split[1])) if n >= 3 else 0
    n_test = max(1, round(n * split[2])) if n >= 3 else 0
    return {'valid': items[:n_val], 'test': items[n_val:n_val + n_test], 'train': items[n_val + n_test:]}


# same, but each group (key) split separately -> every class in every split (small datasets)
def split_by(items, key, split=SPLIT, seed=SEED):
    groups = {}
    for x in items:
        groups.setdefault(key(x), []).append(x)
    out = {s: [] for s in SPLITS}
    for k in sorted(groups, key=str):
        for s, xs in split_list(groups[k], split, seed).items():
            out[s] += xs
    return out


# image -> its rarest class id (-1 = no box). used as split group
def rarest_class(img, index, totals):
    label = find_label(img, index)
    if not label:
        return -1
    ids = set()
    with open(label, encoding='utf-8') as f:
        for line in f:
            v = line.split()
            if v and v[0].lstrip('-').replace('.', '', 1).isdigit():
                ids.add(int(float(v[0])))
    return min(ids, key=lambda c: (totals.get(c, 0), c)) if ids else -1


# same files + same settings -> same fingerprint (used to skip re-training)
# file CONTENT, not modified time -> same value after zip / Colab / Google Drive copy
def fingerprint(src, extra=None):
    h = hashlib.md5()
    files = list_images(src) + [os.path.join(r, f) for r, _, fs in os.walk(src) for f in fs
                                if f.endswith(('.txt', '.yaml', '.names'))]
    for p in sorted(files, key=lambda p: os.path.relpath(p, src).replace('\\', '/')):
        h.update(os.path.relpath(p, src).replace('\\', '/').encode('utf-8') + b'|')
        with open(p, 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 20), b''):
                h.update(chunk)
    h.update(json.dumps(extra or {}, sort_keys=True).encode('utf-8'))
    return h.hexdigest()


# check one label file -> (number of boxes per class, bad lines)
def check_label(path, nc):
    counts, bad = [0] * nc, 0
    with open(path, encoding='utf-8') as f:
        for line in f:
            v = line.split()
            if not v:
                continue
            try:
                c = int(float(v[0]))
                ok = len(v) == 5 and 0 <= c < nc and all(0 <= float(x) <= 1 for x in v[1:])
            except ValueError:
                ok = False
            if ok:
                counts[c] += 1
            else:
                bad += 1
    return counts, bad


# label txt -> dst. merge -> class id 0 for every valid line (stone/metal/plastic -> foreign)
def copy_label(src, dst, nc, merge=None):
    if not merge:
        shutil.copy2(src, dst)
        return
    out = []
    with open(src, encoding='utf-8') as f:
        for line in f:
            v = line.split()
            try:
                if len(v) == 5 and 0 <= int(float(v[0])) < nc:
                    out.append(' '.join(['0'] + v[1:]))
            except ValueError:
                pass
    with open(dst, 'w', encoding='utf-8') as f:
        f.write('\n'.join(out) + ('\n' if out else ''))


# our dataset -> {train,valid,test}/{images,labels} in out_dir
# - rebuilt only when the files changed (done.json)
# - returns (out_dir, class names)
# - merge='foreign' -> every class id becomes 0 (one class). split still uses the original classes
def prepare_custom(src, out_dir, class_names=None, split=SPLIT, seed=SEED, merge=None):
    src, out_dir = os.path.abspath(src), os.path.abspath(out_dir)
    names = list(class_names or read_class_names(src))
    if not names:
        raise ValueError(f'클래스 이름이 없어요: {src} 에 classes.txt (한 줄에 하나) 또는 data.yaml(names) 를 넣어 주세요.')
    if not count_images(src):
        raise ValueError(f'이미지가 없어요: {src}')

    params = {'src': src, 'names': names, 'split': list(split), 'seed': seed, 'fp': fingerprint(src), 'merge': merge}
    marker = os.path.join(out_dir, 'done.json')
    if os.path.exists(marker):
        with open(marker, encoding='utf-8') as f:
            if json.load(f) == params:
                print('dataset path:', out_dir, '(cached)')
                return out_dir, ([merge] if merge else names)
    shutil.rmtree(out_dir, ignore_errors=True)

    index = label_index(src)
    found = collect(src)
    if None in found:   # split by rarest class -> stone / metal / plastic all in valid and test
        totals = {}
        for img in found[None]:
            label = find_label(img, index)
            if label:
                for c, n in enumerate(check_label(label, len(names))[0]):
                    totals[c] = totals.get(c, 0) + n
        imgs = found[None]
        found = split_by(imgs, lambda x: rarest_class(x, index, totals), split, seed)
        if not found['valid'] or not found['test']:   # groups too small -> plain random split
            found = split_list(imgs, split, seed)
    else:
        if not found.get('valid'):
            rest = split_list(found['train'], (0.85, 0.15, 0), seed)
            found['train'], found['valid'] = rest['train'] + rest['test'], rest['valid']
        if not found.get('test'):
            found['test'] = found['valid']   # no test set -> test score = valid score

    counts, bad, n_bg = [0] * len(names), 0, 0
    split_counts = {s: [0] * len(names) for s in SPLITS}
    for s in SPLITS:
        os.makedirs(os.path.join(out_dir, s, 'images'), exist_ok=True)
        os.makedirs(os.path.join(out_dir, s, 'labels'), exist_ok=True)
        for img in found.get(s, []):
            # sub/a.jpg -> sub__a.jpg (same name in two folders must not collide)
            key = os.path.relpath(img, src).replace('\\', '/').replace('/', '__')
            key = os.path.splitext(key)[0]
            shutil.copy2(img, os.path.join(out_dir, s, 'images', key + os.path.splitext(img)[1].lower()))
            label = find_label(img, index)
            if label:
                c, b = check_label(label, len(names))
                counts, bad = [x + y for x, y in zip(counts, c)], bad + b
                split_counts[s] = [x + y for x, y in zip(split_counts[s], c)]
                n_bg += sum(c) == 0
                copy_label(label, os.path.join(out_dir, s, 'labels', key + '.txt'), len(names), merge)
            else:
                n_bg += 1

    print(f'dataset: train {len(found["train"])} / valid {len(found["valid"])} / test {len(found["test"])} 장'
          f'  (정상=라벨 없음 {n_bg}장)')
    print('  boxes:', ', '.join(f'{n}={c}' for n, c in zip(names, counts)))
    for s in SPLITS:
        print(f'    {s:5s}:', ', '.join(f'{n}={c}' for n, c in zip(names, split_counts[s])))
    if bad:
        print(f'  !! 잘못된 라벨 줄 {bad}개는 학습에서 빠져요 (클래스 번호가 classes.txt 줄 수보다 크거나 좌표가 0~1 밖)')
    if not sum(counts):
        raise ValueError('박스가 하나도 없어요. 라벨 txt 이름이 이미지 이름과 같은지, '
                         'YOLO 형식(class cx cy w h, 0~1)인지 확인하세요.')

    with open(marker, 'w', encoding='utf-8') as f:
        json.dump(params, f)
    if merge:
        print(f'  merge: {"+".join(names)} -> {merge} ({sum(counts)} boxes, class 0)')
    print('dataset path:', out_dir)
    return out_dir, ([merge] if merge else names)


# ultralytics needs a data.yaml that points to the dataset folder
def write_data_yaml(data_dir, yaml_path, class_names=('stone',)):
    data = {
        'path': os.path.abspath(data_dir),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'names': dict(enumerate(class_names)),
    }
    os.makedirs(os.path.dirname(yaml_path), exist_ok=True)
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, allow_unicode=True)
    return yaml_path

## 3. Augmentation (albumentations)

In [ ]:
# ultralytics ALWAYS applies its own augmentation too: mosaic, scale/translate, left-right flip, HSV
# (mosaic can be turned off with mosaic=0)
# the lists below are EXTRA albumentations transforms
AUGMENTATIONS = ('default', 'none', 'xray', 'nir', 'strong')


def get_augmentations(name):
    import albumentations as A

    # ultralytics built-in albumentations (Blur, MedianBlur, ToGray, CLAHE, 1% each)
    if name == 'default':
        return None

    # no albumentations
    if name == 'none':
        return []

    # x-ray like changes
    xray = [
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
        A.Blur(blur_limit=3, p=0.1),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
    ]
    if name == 'xray':
        return xray

    # NIR side camera: product always upright -> no vertical flip / 90 rotation
    if name == 'nir':
        return [
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),  # LED / exposure
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            A.GaussNoise(std_range=(0.01, 0.04), p=0.3),                                    # sensor noise
            A.MotionBlur(blur_limit=5, p=0.15),                                             # belt moving
            A.CLAHE(p=0.1),
        ]

    # xray + harder image quality changes
    if name == 'strong':
        return xray + [
            A.CLAHE(p=0.3),
            A.MotionBlur(blur_limit=5, p=0.2),
            A.Downscale(scale_range=(0.5, 0.9), p=0.2),
        ]

    raise ValueError(f'unknown augmentation: {name} (choose from {AUGMENTATIONS})')

## 4. Live training plot and history
After every epoch the six plots below are redrawn in the cell output (three train/val losses, precision and recall, mAP50 and mAP50-95, learning rate).
The values are kept in `LivePlot.history` (one dict per epoch) and saved to `history.csv`, so you can redraw anything after training.

In [ ]:
import pandas as pd
from IPython.display import clear_output

LOSSES = ('box_loss', 'cls_loss', 'dfl_loss')


# trainer -> one row of values for this epoch (works on ultralytics 8.3 and 8.4)
def epoch_row(trainer):
    row = {'epoch': trainer.epoch + 1}
    try:
        row.update(trainer.label_loss_items(trainer.tloss, prefix='train'))
    except Exception:
        t = trainer.tloss
        if isinstance(t, dict):
            row.update({f'train/{k}': float(v) for k, v in t.items()})
    row.update({k: float(v) for k, v in (trainer.metrics or {}).items()})
    row.update({k: float(v) for k, v in (getattr(trainer, 'lr', None) or {}).items()})
    return row


# history (list of dicts) -> 6-panel figure
def draw_history(df, title='', best_col='metrics/mAP50(B)'):
    fig, ax = plt.subplots(2, 3, figsize=(16, 8))
    ax = ax.ravel()
    for k, loss in enumerate(LOSSES):
        for split, style in (('train', 'o-'), ('val', 's--')):
            col = f'{split}/{loss}'
            if col in df:
                ax[k].plot(df['epoch'], df[col], style, ms=3, label=split)
        ax[k].set_title(loss)
        ax[k].legend()
    for col, lab in (('metrics/precision(B)', 'precision'), ('metrics/recall(B)', 'recall')):
        if col in df:
            ax[3].plot(df['epoch'], df[col], 'o-', ms=3, label=lab)
    ax[3].set_ylim(0, 1)
    ax[3].set_title('precision / recall (val)')
    ax[3].legend()
    for col, lab in (('metrics/mAP50(B)', 'mAP50'), ('metrics/mAP50-95(B)', 'mAP50-95')):
        if col in df:
            ax[4].plot(df['epoch'], df[col], 'o-', ms=3, label=lab)
    if best_col in df and len(df):
        b = df[best_col].idxmax()
        ax[4].axvline(df.loc[b, 'epoch'], color='gray', ls=':', label=f"best ep{int(df.loc[b, 'epoch'])}")
    ax[4].set_ylim(0, 1)
    ax[4].set_title('mAP (val)')
    ax[4].legend()
    for col in [c for c in df.columns if c.startswith('lr/')]:
        ax[5].plot(df['epoch'], df[col], '-', label=col)
    ax[5].set_title('learning rate')
    for a in ax:
        a.set_xlabel('epoch')
        a.grid(alpha=0.3)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


# ultralytics callback: each epoch, append to history, redraw, and save live.png / history.csv
# (a plotting error never stops training)
class LivePlot:
    def __init__(self, show=True):
        self.show = show
        self.history = []

    def __call__(self, trainer):
        try:
            row = epoch_row(trainer)
            if self.history and self.history[-1]['epoch'] == row['epoch']:   # final re-validation of best.pt, skip it
                return
            self.history.append(row)
            df = pd.DataFrame(self.history)
            df.to_csv(os.path.join(trainer.save_dir, 'history.csv'), index=False)
            m50 = df.get('metrics/mAP50(B)', pd.Series([0]))
            title = (f"{os.path.basename(str(trainer.save_dir))}  epoch {row['epoch']}/{trainer.epochs}  "
                     f"mAP50 {row.get('metrics/mAP50(B)', 0):.3f} (best {m50.max():.3f})  "
                     f"mAP50-95 {row.get('metrics/mAP50-95(B)', 0):.3f}")
            fig = draw_history(df, title)
            fig.savefig(os.path.join(trainer.save_dir, 'live.png'), dpi=80)
            if self.show:
                clear_output(wait=True)
                display(fig)
            plt.close(fig)
        except Exception as e:
            print('LivePlot skip:', repr(e))


# dataset stats: boxes per class per split, and box sizes
def plot_dataset(data_dir, names, save=None):
    import numpy as np
    counts = {s: [0] * len(names) for s in SPLITS}
    wh = []
    for s in SPLITS:
        for f in glob.glob(os.path.join(data_dir, s, 'labels', '*.txt')):
            for line in open(f, encoding='utf-8'):
                v = line.split()
                if len(v) == 5 and int(float(v[0])) < len(names):
                    counts[s][int(float(v[0]))] += 1
                    wh.append((float(v[3]), float(v[4]), int(float(v[0]))))
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    x = np.arange(len(names))
    for k, s in enumerate(SPLITS):
        ax[0].bar(x + (k - 1) * 0.27, counts[s], 0.27, label=s)
    ax[0].set_xticks(x, names)
    ax[0].set_title('boxes per class / split')
    ax[0].legend()
    if wh:
        w, h, c = zip(*wh)
        sc = ax[1].scatter(w, h, c=c, cmap='viridis', s=14)
        ax[1].legend(sc.legend_elements()[0], names, title='class')
    ax[1].set_xlabel('box width (0~1)')
    ax[1].set_ylabel('box height (0~1)')
    ax[1].set_title('box size')
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=100)
    display(fig)
    plt.close(fig)

## 5. Train, evaluate, save

In [ ]:
# pretrained weights -> {work_dir}/pretrained (downloaded automatically on first use)
def load_model(name, work_dir):
    from ultralytics import YOLO
    if not name.endswith('.pt'):
        os.makedirs(os.path.join(work_dir, 'pretrained'), exist_ok=True)
        name = os.path.join(work_dir, 'pretrained', name + '.pt')
    return YOLO(name)


# 'auto' -> 0 (GPU) or 'cpu'. If a GPU was asked for but there isn't one, fall back to cpu and say so
def pick_device(device='auto'):
    import torch
    if torch.cuda.is_available():
        if device in ('auto', None, ''):
            device = 0
        print('device:', device, torch.cuda.get_device_name(0))
        return device
    if str(device).lower() != 'cpu':
        print('!! GPU 가 안 보여요 -> CPU 로 학습해요 (아주 느림).')
        print('   런타임 -> 런타임 유형 변경 -> T4 GPU 로 바꾸고, 런타임 다시 시작 후 처음부터 실행하세요.')
    return 'cpu'


# cfg -> trained model (with best.pt loaded), run name, data.yaml
def train(cfg):
    from ultralytics import settings
    cfg['device'] = pick_device(cfg.get('device', 'auto'))
    name = cfg['name'] or f"{cfg['model']}_{cfg['aug']}_{datetime.now():%Y%m%d_%H%M%S}"
    runs = os.path.join(cfg['work_dir'], 'runs')

    # data
    src = unzip_all(cfg['root'], cfg['work_dir'])
    data_dir, names = prepare_custom(src, os.path.join(cfg['work_dir'], 'dataset_yolo'),
                                     split=tuple(cfg['split']), seed=cfg['split_seed'],
                                     merge=cfg.get('merge'))
    data_yaml = write_data_yaml(data_dir, os.path.join(runs, name, 'data.yaml'), names)
    print('classes:', names)

    # model
    settings.update({'tensorboard': False})
    model = load_model(cfg['model'], cfg['work_dir'])
    live = LivePlot(show=cfg.get('live', True))
    model.add_callback('on_fit_epoch_end', live)

    # train
    train_args = dict(
        data=data_yaml,
        epochs=cfg['epochs'],
        batch=cfg['batch'],
        imgsz=cfg['imgsz'],
        optimizer=cfg['optimizer'],
        lr0=cfg['lr'],
        patience=cfg['patience'],
        device=cfg['device'],
        workers=cfg['workers'],
        seed=cfg['seed'],
        mosaic=cfg['mosaic'],
        project=runs,
        name=name,
        exist_ok=True,
    )
    aug = get_augmentations(cfg['aug'])
    if aug is not None:   # None = ultralytics defaults
        train_args['augmentations'] = aug
    train_args.update(cfg['opt'])   # opt wins
    model.train(**train_args)
    return model, name, data_yaml, names, data_dir, live


# score on the test split
def evaluate(model, cfg, data_yaml, name):
    metrics = model.val(data=data_yaml, split='test', imgsz=cfg['imgsz'], batch=cfg['batch'],
                        device=cfg['device'], project=os.path.join(cfg['work_dir'], 'runs'),
                        name=name + '_test', exist_ok=True)
    print(f'test  mAP@50: {metrics.box.map50:.4f}  mAP@50:95: {metrics.box.map:.4f}')
    return metrics


# the four files the conveyor reads -> out_dir (Drive)
#   detect.pt        TorchScript, loaded by ai/app.py with the default MODEL_BACKEND=torchscript
#   best.pt          ultralytics checkpoint (for MODEL_BACKEND=ultralytics)
#   classes.txt      class names (line order = class id = defectClass)
#   model_info.json  training info (ai/app.py reads imgsz from here)
def save_for_conveyor(model, cfg, names, name, metrics):
    from ultralytics import YOLO
    out = cfg['out_dir']
    os.makedirs(out, exist_ok=True)
    best = str(model.trainer.best)
    ts = YOLO(best).export(format='torchscript', imgsz=cfg['imgsz'], device='cpu')
    shutil.copy(ts, os.path.join(out, 'detect.pt'))
    shutil.copy(best, os.path.join(out, 'best.pt'))
    with open(os.path.join(out, 'classes.txt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(names) + '\n')
    info = {'time': f'{datetime.now():%Y-%m-%d %H:%M}', 'name': name, 'model': cfg['model'],
            'imgsz': cfg['imgsz'], 'epochs': cfg['epochs'], 'aug': cfg['aug'], 'classes': names,
            'modelVer': cfg['model_ver'], 'torchscript': True,
            'mAP50': round(metrics.box.map50, 4), 'mAP50_95': round(metrics.box.map, 4)}
    with open(os.path.join(out, 'model_info.json'), 'w', encoding='utf-8') as f:
        json.dump(info, f, ensure_ascii=False, indent=2)

    # also keep the training record for the judges: plots, confusion matrix, results.csv
    run_dir = str(model.trainer.save_dir)
    rec = os.path.join(out, 'record_' + name)
    os.makedirs(rec, exist_ok=True)
    for f in ('results.png', 'results.csv', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
              'labels.jpg', 'live.png', 'history.csv', 'args.yaml', 'data.yaml'):
        for p in (os.path.join(run_dir, f), os.path.join(os.path.dirname(run_dir), name, f)):
            if os.path.exists(p):
                shutil.copy(p, rec)
                break
    print('saved:', out)
    for f in sorted(os.listdir(out)):
        print('  ', f)
    return info


# everything: data -> train -> test -> save
def run(cfg):
    model, name, data_yaml, names, data_dir, live = train(cfg)
    metrics = evaluate(model, cfg, data_yaml, name)
    info = save_for_conveyor(model, cfg, names, name, metrics)
    run_dir = str(model.trainer.save_dir)
    return dict(model=model, name=name, data_yaml=data_yaml, data_dir=data_dir, names=names,
                metrics=metrics, info=info, run_dir=run_dir,
                test_dir=os.path.join(cfg['work_dir'], 'runs', name + '_test'),
                record_dir=os.path.join(cfg['out_dir'], 'record_' + name),
                history=pd.DataFrame(live.history))

## 6. Check the data (once, before training)
Unzips the data and shows the train / valid / test split and the box count per class.

In [ ]:
src = unzip_all(CFG['root'], CFG['work_dir'])
data_dir, names = prepare_custom(src, os.path.join(CFG['work_dir'], 'dataset_yolo'),
                                 split=tuple(CFG['split']), seed=CFG['split_seed'],
                                 merge=CFG.get('merge'))
print('classes:', names)
plot_dataset(data_dir, names)

## 7. Train and save to `ROOT/model/`

In [ ]:
result = run(CFG)
result['info']

## 8. Plots from the history
Redraws from `result['history']` (or a saved `history.csv`) after training. The plots are also saved under `ROOT/model/record_*/`.
If the runtime disconnected, load `pd.read_csv('.../record_<name>/history.csv')` and call the same functions.

In [ ]:
# loss / P / R / mAP / lr per epoch
def plot_history(history, title='', save=None):
    df = pd.DataFrame(history)
    fig = draw_history(df, title)
    if save:
        fig.savefig(save, dpi=100)
    display(fig)
    plt.close(fig)


# best epoch summary, plus mAP50 vs loss to spot overfitting
def plot_best(history, key='metrics/mAP50(B)', save=None):
    df = pd.DataFrame(history)
    b = df[key].idxmax()
    cols = ['epoch', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']
    print('best epoch:')
    display(df.loc[[b], [c for c in cols if c in df]].round(4))
    fig, ax = plt.subplots(figsize=(8, 4))
    tr = df[[c for c in df if c.startswith('train/')]].sum(axis=1)
    va = df[[c for c in df if c.startswith('val/')]].sum(axis=1)
    ax.plot(df['epoch'], tr, label='train loss (sum)')
    ax.plot(df['epoch'], va, label='val loss (sum)')
    ax2 = ax.twinx()
    ax2.plot(df['epoch'], df[key], 'g-', label='mAP50')
    ax2.set_ylim(0, 1)
    ax.axvline(df.loc[b, 'epoch'], color='gray', ls=':')
    ax.set_xlabel('epoch')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    ax.set_title('loss vs mAP50 (val loss rising again = overfitting)')
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=100)
    display(fig)
    plt.close(fig)


# per-class AP50 / AP50-95 on the test split
def plot_per_class(metrics, names, save=None):
    import numpy as np
    idx = list(metrics.box.ap_class_index)
    ap50 = [float(metrics.box.ap50[idx.index(i)]) if i in idx else 0 for i in range(len(names))]
    ap = [float(metrics.box.ap[idx.index(i)]) if i in idx else 0 for i in range(len(names))]
    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - 0.2, ap50, 0.4, label='AP50')
    ax.bar(x + 0.2, ap, 0.4, label='AP50-95')
    for i, v in enumerate(ap50):
        ax.text(i - 0.2, v + 0.02, f'{v:.2f}', ha='center')
    ax.set_xticks(x, names)
    ax.set_ylim(0, 1.1)
    ax.set_title(f'test per class  (mAP50 {metrics.box.map50:.3f}, mAP50-95 {metrics.box.map:.3f})')
    ax.legend()
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=100)
    display(fig)
    plt.close(fig)
    display(pd.DataFrame({'class': names, 'AP50': ap50, 'AP50-95': ap}).round(4))


# plots ultralytics made during the test run (PR / F1 curves, confusion matrix)
def show_val_curves(test_dir, width=520):
    pats = ('*PR_curve*.png', '*F1_curve*.png', 'confusion_matrix_normalized.png')
    for pat in pats:
        for p in sorted(glob.glob(os.path.join(test_dir, pat))):
            print(os.path.basename(p))
            display(ShowImage(p, width=width))


# predicted boxes on some test images
def show_predictions(weights, data_dir, imgsz, n=4, conf=0.5, out_dir='/content/pred'):
    from ultralytics import YOLO
    os.makedirs(out_dir, exist_ok=True)
    imgs = sorted(glob.glob(os.path.join(data_dir, 'test', 'images', '*')))[:n]
    for r in YOLO(weights).predict(imgs, imgsz=imgsz, conf=conf, verbose=False):
        p = os.path.join(out_dir, os.path.basename(r.path))
        r.save(filename=p)
        display(ShowImage(p, width=600))

In [ ]:
rec = result['record_dir']
plot_history(result['history'], result['name'], save=os.path.join(rec, 'history.png'))
plot_best(result['history'], save=os.path.join(rec, 'best_epoch.png'))
plot_per_class(result['metrics'], result['names'], save=os.path.join(rec, 'per_class.png'))

In [ ]:
show_val_curves(result['test_dir'])
show_predictions(os.path.join(CFG['out_dir'], 'best.pt'), result['data_dir'], CFG['imgsz'])

## 9. Put the model on the conveyor

Download these from `ROOT/model/` on Drive into the project's `models/` folder, overwriting what's there.

| File | Goes to | Needed |
|---|---|---|
| `detect.pt` | `models/detect.pt` | yes |
| `classes.txt` | `models/classes.txt` | yes |
| `model_info.json` | `models/model_info.json` | recommended (sets the input size) |
| `best.pt` | `models/best.pt` | optional |

Then start `run_all.bat`. If the AI log shows `모델 로드: detect.pt (torchscript) 클래스={0: 'foreign'}`, it worked.
The `record_*/` folder (plots, confusion matrix, results.csv) is useful material for the judges.